In [2]:
import os
import pandas as pd

# Path to your MRI image folder
MRI_IMAGE_DIR = r"C:\Users\yuvar\OneDrive\Desktop\Testing"

# Define label names
LABELS = ['epidural', 'intraparenchymal', 'intraventricular', 'subarachnoid', 'subdural']

# Collect all image paths and labels
data = []

for label in LABELS:
    label_dir = os.path.join(MRI_IMAGE_DIR, label)
    if not os.path.exists(label_dir):
        continue
    for img_name in os.listdir(label_dir):
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(label_dir, img_name)
            # Create label row: 1 for current class, 0 for others
            label_row = {lbl: 1 if lbl == label else 0 for lbl in LABELS}
            label_row['image_path'] = img_path
            label_row['modality'] = 'MRI'   # Add modality for combined model later
            data.append(label_row)

# Convert to DataFrame
df_mri = pd.DataFrame(data)

# Reorder columns
df_mri = df_mri[['image_path', 'modality'] + LABELS]

# Save to CSV
df_mri.to_csv("mri_labels_no.csv", index=False)
print("✅ MRI CSV saved as mri_labels.csv")
print(df_mri.head())


✅ MRI CSV saved as mri_labels.csv
                                          image_path modality  epidural  \
0  C:\Users\yuvar\OneDrive\Desktop\Testing\epidur...      MRI         1   
1  C:\Users\yuvar\OneDrive\Desktop\Testing\epidur...      MRI         1   
2  C:\Users\yuvar\OneDrive\Desktop\Testing\epidur...      MRI         1   
3  C:\Users\yuvar\OneDrive\Desktop\Testing\epidur...      MRI         1   
4  C:\Users\yuvar\OneDrive\Desktop\Testing\epidur...      MRI         1   

   intraparenchymal  intraventricular  subarachnoid  subdural  
0                 0                 0             0         0  
1                 0                 0             0         0  
2                 0                 0             0         0  
3                 0                 0             0         0  
4                 0                 0             0         0  


In [6]:
import os
import pandas as pd

# Path to your MRI image folder
MRI_IMAGE_DIR = r"C:\Users\yuvar\OneDrive\Desktop\Testing"

# Define label names
LABELS = ['epidural', 'intraparenchymal', 'intraventricular', 'subarachnoid', 'subdural']

# Collect all image paths and labels
data = []

print("🔍 Collecting labeled images...")

# Step 1: Load hemorrhage images
for label in LABELS:
    label_dir = os.path.join(MRI_IMAGE_DIR, label)
    if not os.path.exists(label_dir):
        print(f"⚠️ Skipping {label} — folder not found")
        continue

    for img_name in os.listdir(label_dir):
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(label_dir, img_name)
            label_row = {lbl: 1 if lbl == label else 0 for lbl in LABELS}
            label_row['image_path'] = img_path
            label_row['modality'] = 'MRI'
            data.append(label_row)

print(f"✅ Found {len(data)} labeled images so far")

# Step 2: Load "no" folder (all zeros)
no_dir = os.path.join(MRI_IMAGE_DIR, 'no')
if os.path.exists(no_dir):
    print(f"🧩 Adding images from: {no_dir}")
    count_no = 0
    for img_name in os.listdir(no_dir):
        if img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            img_path = os.path.join(no_dir, img_name)
            label_row = {lbl: 0 for lbl in LABELS}
            label_row['image_path'] = img_path
            label_row['modality'] = 'MRI'
            data.append(label_row)
            count_no += 1
    print(f"✅ Added {count_no} 'no' images")
else:
    print(f"🚫 'no' folder not found at: {no_dir}")

# Step 3: Save to CSV
if data:
    df_mri = pd.DataFrame(data)
    df_mri = df_mri[['image_path', 'modality'] + LABELS]
    df_mri.to_csv("mri_labels_no.csv", index=False)
    print(f"✅ MRI CSV saved as mri_label_nos.csv with {len(df_mri)} rows total")
    print(df_mri.tail())
else:
    print("🚫 No images found at all!")


🔍 Collecting labeled images...
✅ Found 1540 labeled images so far
🧩 Adding images from: C:\Users\yuvar\OneDrive\Desktop\Testing\no
✅ Added 98 'no' images
✅ MRI CSV saved as mri_label_nos.csv with 1638 rows total
                                             image_path modality  epidural  \
1633  C:\Users\yuvar\OneDrive\Desktop\Testing\no\No1...      MRI         0   
1634  C:\Users\yuvar\OneDrive\Desktop\Testing\no\No1...      MRI         0   
1635  C:\Users\yuvar\OneDrive\Desktop\Testing\no\No2...      MRI         0   
1636  C:\Users\yuvar\OneDrive\Desktop\Testing\no\No2...      MRI         0   
1637  C:\Users\yuvar\OneDrive\Desktop\Testing\no\No2...      MRI         0   

      intraparenchymal  intraventricular  subarachnoid  subdural  
1633                 0                 0             0         0  
1634                 0                 0             0         0  
1635                 0                 0             0         0  
1636                 0                 0           

In [ ]:
import pandas as pd

# Load your CT and MRI CSVs
ct_df = pd.read_csv("../data/stage_2_train_pivot.csv")
mri_df = pd.read_csv("../tocsv/mri_labels.csv")

# Fix CT column names
ct_df = ct_df.rename(columns={'image_id': 'image_path'})
if 'any' in ct_df.columns:
    ct_df = ct_df.drop(columns=['any'])  # optional, depending on MRI labels
ct_df['modality'] = 'CT'

# Ensure consistent column order
common_cols = ['image_path', 'modality', 'epidural', 'intraparenchymal', 
               'intraventricular', 'subarachnoid', 'subdural']

# Align MRI columns if needed
if 'modality' not in mri_df.columns:
    mri_df['modality'] = 'MRI'

mri_df = mri_df[common_cols]
ct_df = ct_df[common_cols]

# Merge both datasets
combined_df = pd.concat([ct_df, mri_df], ignore_index=True)

# Save final merged file
combined_df.to_csv("combined_labels.csv", index=False)

print("✅ Combined CT+MRI labels saved as combined_labels_.csv")
print("Total samples:", len(combined_df))


✅ Combined CT+MRI labels saved as combined_labels.csv
Total samples: 754343
